# L06 Assignment — Two Forecasting Problems

Two focused exercises in different domains. Apply the L06 toolkit (decomposition + classical + ML-with-lags) to each.

1. **Part A — UK electricity demand.** Strong annual + weekly + daily seasonality. Forecast the next week.
2. **Part B — Coffee-shop daily customers.** Smaller, noisier, weekly seasonality only. Forecast the next month.

**Sample solutions** at the bottom. Attempt each before scrolling.

**Time budget:** ~75 minutes.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4.5)
print("✅ Setup complete.")

---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Part A — UK electricity demand (STL + ETS + ML-with-lags) |
| **🔵 Advanced Track** | Learners with prior ML background | Part B — coffee-shop daily customers (open: pick a method, beat baseline by 5%) |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


# Part A — UK Electricity Demand

A national grid operator hands you 2 years of daily electricity demand. They need a 7-day-ahead forecast every Monday. **Your goal: build the best ML-with-lags forecaster you can on this data.**

## Generate the electricity dataset

In [ ]:
def generate_electricity_data(start="2024-01-01", end="2025-12-31", seed=2026):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    n = len(dates)
    doy = dates.dayofyear.values
    dow = dates.dayofweek.values

    base = 32000   # MWh/day baseline
    # Strong annual cycle: winter peak (cold heating), summer secondary peak (air-con)
    annual_winter = 5500 * np.cos(2 * np.pi * (doy - 15) / 365.25)
    annual_summer = 1500 * np.maximum(np.cos(2 * np.pi * (doy - 200) / 365.25), 0)
    # Weekly cycle: weekdays > weekends
    weekly = np.where(dow < 5, 1800, -1500)
    # Small upward trend (electrification)
    trend = np.arange(n) * 2.0
    noise = rng.normal(0, 700, size=n)
    demand = base + annual_winter + annual_summer + weekly + trend + noise

    return pd.DataFrame({"date": dates, "demand_mwh": np.maximum(demand, 15000).round(0)})

elec = generate_electricity_data().set_index("date")
print(f"Electricity dataset: {len(elec):,} days")
print(f"Demand range: {elec['demand_mwh'].min():,.0f} - {elec['demand_mwh'].max():,.0f} MWh/day")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(elec.index, elec["demand_mwh"], linewidth=0.7, color="steelblue")
ax.set_ylabel("Demand (MWh)"); ax.set_title("UK daily electricity demand")
plt.tight_layout(); plt.show()

## Exercise A1 — Decompose the series

**Tasks:**
1. Run STL on `elec["demand_mwh"]` with the right `period` for daily data with weekly seasonality.
2. Plot the 4-panel decomposition.
3. State what you see in the trend component (one sentence) and what you see in the seasonal (one sentence).

*Your code:*

In [ ]:
# Exercise A1 — STL decomposition

# 1. Run STL. Daily data with a weekly cycle → period = 7.
stl_elec = STL(elec["demand_mwh"], period=7, robust=True).fit()

# 2. Plot the 4-panel decomposition (observed / trend / seasonal / residual).
fig = stl_elec.plot()
fig.set_size_inches(13, 8)
plt.tight_layout()
plt.show()

# 3. Quick numeric peek to back up the observations.
print(f"Trend range:    {stl_elec.trend.min():,.0f} - {stl_elec.trend.max():,.0f} MWh")
print(f"Seasonal range: {stl_elec.seasonal.min():,.0f} - {stl_elec.seasonal.max():,.0f} MWh")

*Your observations:*

> (your answer here)

## Exercise A2 — ETS-weekly baseline

**Tasks:**
1. Hold out the last 7 days as the test set.
2. Fit `ExponentialSmoothing(trend="add", seasonal="add", seasonal_periods=7)` on the training data.
3. Forecast 7 days; compute MAE + MAPE.

*Your code:*

In [ ]:
# Exercise A2
# (your code here)


## Exercise A3 — ML-with-lags

**Tasks:**
1. Build features: `lag_1`, `lag_7`, `lag_365`, `dayofweek`, `month`, `dayofyear`.
2. Drop NaN rows.
3. Split on the same date boundary as A2 (last 7 days = test).
4. Train `HistGradientBoostingRegressor` on the resulting tabular data.
5. Forecast 7 days using one-step-ahead (lag_1 is actual yesterday for each test day).
6. Compute MAE + MAPE.

*Your code:*

In [ ]:
# Exercise A3
# (your code here)


## Exercise A4 — Which method wins?

**Tasks:**
1. Compare A2 and A3 results side by side.
2. State which method wins on MAE.
3. Speculate (1 sentence) on WHY the winner won.

*Your answer:*

> (your answer here)

---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


# Part B — Coffee-Shop Daily Customers

A neighbourhood coffee shop wants to predict daily customer count to plan staffing. The shop has 18 months of data. Smaller scale, noisier, weekly seasonality only.

## Generate the coffee-shop dataset

In [ ]:
def generate_coffee_data(start="2024-07-01", end="2025-12-31", seed=2027):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    n = len(dates)
    dow = dates.dayofweek.values
    doy = dates.dayofyear.values

    base = 220  # customers per day
    # Weekly: Saturday peak, Sunday slightly lower
    weekly_lift = np.where(dow == 5, 100,
                  np.where(dow == 6, 70,
                  np.where(dow == 4, 30, 0)))
    # Mild summer dip (people on holiday, staff also reduced)
    summer_dip = -25 * np.maximum(np.cos(2 * np.pi * (doy - 200) / 365.25), 0)
    # Weather noise (substantial)
    noise = rng.normal(0, 25, size=n)
    customers = base + weekly_lift + summer_dip + noise

    return pd.DataFrame({"date": dates, "customers": np.maximum(customers, 30).round().astype(int)})

coffee = generate_coffee_data().set_index("date")
print(f"Coffee dataset: {len(coffee):,} days")
print(f"Customers range: {coffee['customers'].min()} - {coffee['customers'].max()}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(coffee.index, coffee["customers"], linewidth=0.8, color="coral")
ax.set_ylabel("Customers / day"); ax.set_title("Coffee-shop daily customer count")
plt.tight_layout(); plt.show()

## Exercise B1 — Naive baseline

**Tasks:**
1. Hold out the last 30 days as the test set.
2. Build a Seasonal Naive forecast (predict same-day-last-week).
3. Compute MAE + MAPE.

*Your code:*

In [ ]:
# Exercise B1
# (your code here)


## Exercise B2 — Pick ONE method and beat the baseline

**Task:** train any method from L06 (ETS, ML-with-lags, both) and beat Exercise B1's MAE by at least 5%.

Hints:
- Coffee data has NO annual seasonality (only 18 months of data; weekly cycle dominates). Don't bother with `lag_365`.
- Weekly seasonality dominates → `seasonal_periods=7` for ETS; `lag_7` essential for ML.

*Your code:*

In [ ]:
# Exercise B2 — beat B1 by ≥ 5%
# (your code here)


*State your method, MAE, and how much you beat B1 by:*

> (your answer here)

## ✅ Submission checklist

- [ ] Exercise A1: STL decomposition + 1-sentence observation each on trend and seasonal
- [ ] Exercise A2: ETS-weekly forecast for 7 days + MAE + MAPE
- [ ] Exercise A3: ML-with-lags forecast for 7 days + MAE + MAPE
- [ ] Exercise A4: state the winner + 1-sentence WHY
- [ ] Exercise B1: Seasonal Naive baseline for 30 days + MAE + MAPE
- [ ] Exercise B2: pick a method that beats B1 by ≥5%

---

# 📚 Sample solutions

*Compare to your work AFTER attempting.*

## Sample — Exercise A1 (STL on electricity)

In [ ]:
# === Sample A1 ===

stl_elec = STL(elec["demand_mwh"], period=7, robust=True).fit()
fig = stl_elec.plot()
fig.set_size_inches(13, 8)
plt.tight_layout(); plt.show()

print("Trend observation: Strong winter peak + smaller summer peak; gentle upward trend overall.")
print("Seasonal observation: Weekdays consistently HIGHER than weekends (commercial/industrial demand)." )

## Sample — Exercise A2 (ETS on electricity)

In [ ]:
# === Sample A2 ===

y_elec = elec["demand_mwh"]
TEST = 7
y_tr_e = y_elec.iloc[:-TEST]
y_te_e = y_elec.iloc[-TEST:]

ets_e = ExponentialSmoothing(y_tr_e, trend="add", seasonal="add", seasonal_periods=7).fit()
ets_pred = ets_e.forecast(steps=TEST)
ets_pred.index = y_te_e.index

mae_e = mean_absolute_error(y_te_e, ets_pred)
mape_e = (np.abs(y_te_e - ets_pred) / y_te_e).mean() * 100
print(f"ETS-weekly: MAE = {mae_e:,.0f} MWh, MAPE = {mape_e:.2f}%")

## Sample — Exercise A3 (ML-with-lags on electricity)

In [ ]:
# === Sample A3 ===

elec_feat = pd.DataFrame(index=elec.index)
elec_feat["target"]    = y_elec
elec_feat["lag_1"]     = y_elec.shift(1)
elec_feat["lag_7"]     = y_elec.shift(7)
elec_feat["lag_365"]   = y_elec.shift(365)
elec_feat["dayofweek"] = y_elec.index.dayofweek
elec_feat["month"]     = y_elec.index.month
elec_feat = elec_feat.dropna()

split = y_te_e.index[0]
train_e = elec_feat.loc[elec_feat.index < split]
test_e  = elec_feat.loc[elec_feat.index >= split]
Xtr, ytr = train_e.drop(columns="target"), train_e["target"]
Xte, yte = test_e.drop(columns="target"),  test_e["target"]

ml = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)
ml.fit(Xtr, ytr)
ml_pred = ml.predict(Xte)

mae_ml_e = mean_absolute_error(yte, ml_pred)
mape_ml_e = (np.abs(yte - ml_pred) / yte).mean() * 100
print(f"ML-with-lags one-step: MAE = {mae_ml_e:,.0f} MWh, MAPE = {mape_ml_e:.2f}%")

## Sample — Exercise A4 (comparison)

In [ ]:
# === Sample A4 ===

print(f"ETS-weekly:  MAE {mae_e:,.0f}  MAPE {mape_e:.2f}%")
print(f"ML-w-lags:   MAE {mae_ml_e:,.0f}  MAPE {mape_ml_e:.2f}%")

winner = "ML-w-lags" if mae_ml_e < mae_e else "ETS-weekly"
print(f"\nWinner: {winner}")
print()
if winner == "ETS-weekly":
    print("WHY (likely reason):")
    print("  The 7-day test window is too short for the ML-with-lags trick to shine.")
    print("  ETS-weekly's smoothed trend + weekly seasonality fits a short non-holiday window perfectly.")
    print("  ML's `lag_365` advantage shows up on longer horizons or when the test crosses a regime change.")
else:
    print("WHY (likely reason):")
    print("  ML uses lag_365 to know what last year's same day looked like —")
    print("  captures the annual cycle that ETS-weekly can't see with period=7.")

## Sample — Exercise B1 (Seasonal Naive on coffee shop)

In [ ]:
# === Sample B1 ===

y_coffee = coffee["customers"]
TEST_C = 30
y_tr_c = y_coffee.iloc[:-TEST_C]
y_te_c = y_coffee.iloc[-TEST_C:]

snaive_c = pd.Series(index=y_te_c.index, dtype=float)
for date in y_te_c.index:
    lw = date - pd.Timedelta(days=7)
    snaive_c.loc[date] = y_tr_c.loc[lw] if lw in y_tr_c.index else y_tr_c.iloc[-7:].mean()

mae_snaive = mean_absolute_error(y_te_c, snaive_c)
mape_snaive = (np.abs(y_te_c - snaive_c) / y_te_c).mean() * 100
print(f"Seasonal Naive (coffee): MAE = {mae_snaive:.1f} customers, MAPE = {mape_snaive:.2f}%")

## Sample — Exercise B2 (beat B1)

In [ ]:
# === Sample B2 ===

# ML-with-lags approach
coffee_feat = pd.DataFrame(index=coffee.index)
coffee_feat["target"]    = y_coffee
coffee_feat["lag_1"]     = y_coffee.shift(1)
coffee_feat["lag_7"]     = y_coffee.shift(7)
coffee_feat["lag_14"]    = y_coffee.shift(14)   # additional lag
coffee_feat["dayofweek"] = y_coffee.index.dayofweek
coffee_feat["month"]     = y_coffee.index.month
coffee_feat = coffee_feat.dropna()

split_c = y_te_c.index[0]
train_c = coffee_feat.loc[coffee_feat.index < split_c]
test_c  = coffee_feat.loc[coffee_feat.index >= split_c]
Xtr_c, ytr_c = train_c.drop(columns="target"), train_c["target"]
Xte_c, yte_c = test_c.drop(columns="target"),  test_c["target"]

ml_c = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)
ml_c.fit(Xtr_c, ytr_c)
ml_pred_c = ml_c.predict(Xte_c)

mae_ml_c = mean_absolute_error(yte_c, ml_pred_c)
mape_ml_c = (np.abs(yte_c - ml_pred_c) / yte_c).mean() * 100

print(f"ML-with-lags (coffee): MAE = {mae_ml_c:.1f}, MAPE = {mape_ml_c:.2f}%")
print(f"vs B1 Seasonal Naive:  MAE = {mae_snaive:.1f}")
improvement = (mae_snaive - mae_ml_c) / mae_snaive * 100
print(f"Improvement over B1: {improvement:+.1f}%")

## What's next

You've now forecast two different domains. The L06 pipeline is:
1. **Decompose** to understand structure
2. **Try classical baselines** (Naive, Seasonal Naive, ETS)
3. **Try ML-with-lags** when seasonality is multi-period
4. **Cross-validate** with `TimeSeriesSplit`
5. **Pick the winner on cross-validated metric**, not a single test window

**Next session → L07 (Neural Networks & Deep Learning).** Marcus's question — *"predict checkout completion FROM SEQUENTIAL BEHAVIOUR"* — opens the deep learning toolkit.